<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-12/AI_Modul_12.2_Praktikum_Arsitektur_CNN.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 12.2: Praktikum Arsitektur CNN (VGG vs ResNet)
**Tujuan Pembelajaran:**
1. Membangun modul Residual BasicBlock dengan mekanisme skip connection di PyTorch.
2. Merakit arsitektur ResNet-18 Mini dan membandingkannya terhadap Plain CNN (tanpa residual).
3. Mensimulasikan klasifikasi 4 tingkat kematangan tandan buah segar (TBS) kelapa sawit.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch siap digunakan: {torch.__version__}")

### Bagian 1: Definisi Komparasi Model (Plain CNN vs Residual CNN)

In [ ]:
# Blok Residual Berstandar ResNet
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        
    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual # Shortcut connection
        return self.relu(out)

# Blok Plain (Tanpa Shortcut Connection)
class PlainBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        return out

class MiniNetwork(nn.Module):
    def __init__(self, use_residual=True, num_classes=4):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        block_type = ResBlock if use_residual else PlainBlock
        self.stage = nn.Sequential(
            block_type(32),
            block_type(32),
            block_type(32)
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(32, num_classes)
        )
        
    def forward(self, x):
        return self.head(self.stage(self.stem(x)))

net_res = MiniNetwork(use_residual=True)
net_plain = MiniNetwork(use_residual=False)
print(f"Parameter ResNet: {sum(p.numel() for p in net_res.parameters()):,}")
print(f"Parameter Plain:  {sum(p.numel() for p in net_plain.parameters()):,}")

### Bagian 2: Sintesis Citra Profil Spektral Kematangan Buah Sawit (4 Fraksi)

In [ ]:
def generate_oil_palm_fruit_dataset(n_samples=400):
    X = np.zeros((n_samples, 3, 32, 32), dtype=np.float32)
    y = np.zeros(n_samples, dtype=np.int64)
    
    for i in range(n_samples):
        label = i % 4
        y[i] = label
        if label == 0: # Mentah (Hitam Keunguan)
            r, g, b = 0.20, 0.15, 0.25
        elif label == 1: # Mengkal (Cokelat Kekuningan)
            r, g, b = 0.55, 0.40, 0.15
        elif label == 2: # Matang Sempurna (Oranye Kemerahan Optimal)
            r, g, b = 0.85, 0.35, 0.05
        else: # Lewat Matang (Merah Gelap Kusam)
            r, g, b = 0.50, 0.15, 0.05
            
        noise = np.random.normal(0, 0.04, (3, 32, 32))
        X[i, 0, :, :] = r + noise[0]
        X[i, 1, :, :] = g + noise[1]
        X[i, 2, :, :] = b + noise[2]
        
    X = np.clip(X, 0.0, 1.0)
    return torch.tensor(X), torch.tensor(y)

X_data, y_data = generate_oil_palm_fruit_dataset(400)
dataset = TensorDataset(X_data, y_data)
train_ds, val_ds = torch.utils.data.random_split(dataset, [320, 80])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
print(f"Dataset TBS Sawit berhasil dibuat: {len(train_ds)} train, {len(val_ds)} val.")

### Bagian 3: Pelatihan dan Evaluasi Komparatif (ResNet vs Plain CNN)

In [ ]:
def train_eval(model, epochs=8):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.002)
    losses = []
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for bx, by in train_loader:
            optimizer.zero_grad()
            out = model(bx)
            loss = criterion(out, by)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * bx.size(0)
        losses.append(total_loss / len(train_loader.dataset))
        
    model.eval()
    correct = 0
    with torch.no_grad():
        for vx, vy in val_loader:
            preds = model(vx).argmax(dim=1)
            correct += (preds == vy).sum().item()
    acc = correct / len(val_loader.dataset)
    return losses, acc

print("Melatih ResNet Block Model...")
loss_res, acc_res = train_eval(net_res)
print(f"[HASIL ResNet] Val Accuracy: {acc_res*100:.2f}%")

print("\nMelatih Plain Block Model...")
loss_plain, acc_plain = train_eval(net_plain)
print(f"[HASIL Plain]  Val Accuracy: {acc_plain*100:.2f}%")

# Simpan kurva komparasi
plt.figure(figsize=(7, 4))
plt.plot(loss_res, 'r-o', label=f'ResNet Block (Val Acc: {acc_res*100:.1f}%)')
plt.plot(loss_plain, 'b--s', label=f'Plain Block (Val Acc: {acc_plain*100:.1f}%)')
plt.title('Komparasi Konvergensi Loss: ResNet vs Plain CNN')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig('praktikum_resnet_vs_plain_12_2.png', dpi=200)
plt.close()
print("[OK] Grafik disimpan sebagai 'praktikum_resnet_vs_plain_12_2.png'")